In [0]:
import pandas as pd

### Load & Inspect the Dataset

In [0]:
#Load CSV with sep and decimal arguments
#df = spark.table("workspace.default.bright_coffee_shop_sales_raw_data").toPandas()
df = pd.read_csv("/Workspace/Users/majamp0@gmail.com/1779732970357_Bright_Coffee_Shop_Sales.csv", sep=";", decimal=",")

df.head(100)

In [0]:
#Print the shape
print(df.shape)

In [0]:
#Show all columns
df.columns.tolist()

In [0]:
#Data types and a count of non-null
df.info()

In [0]:
#Checking for missing values and summming them
df.isnull().sum()

In [0]:
#Statistical summarry of numeric columns 
df.describe()

In [0]:
#Summarize columns with object dtype
df.describe(include="object")

### observation of the summary statistics
Hell's Kitchen has the most frequent transactions of 50735. 
The top contributors to the frequency of the transactions in this store are:
Coffee, Brewed Chai Tea and Chocolate Croissant respectively. What can be done to boost the Chocolate Croissant frequency is to coombimne it with one both Coffee and Brewed Chai Tea as a combo.

### Feature Engineering & Distributions

In [0]:
#Adding revenue column 
df["revenue"] = df["unit_price"]*df["transaction_qty"]
df.head()

In [0]:
#Change the date column data type from object to date
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

In [0]:
#Extracting hour and month from transaction_time
df["hour"] = pd.to_datetime(df["transaction_time"]).dt.hour
df["month"] = df["transaction_date"].dt.month

df.head()

In [0]:
#Removal of any duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

importing libraries that will enable us to visualize the data

In [0]:
#Plotting the distribution with KDE
import matplotlib.pyplot as plt
import seaborn as sns

The shape of the shows that highest transactions happened when the unit price was between 0 and 10, which suggest that the lower the price the higher the transactions. Very few transactions occured as the unit price increased.

In [0]:
sns.histplot(df, x="unit_price", kde=True, color="steelblue")
plt.title("Unit Price Distribution")
plt.tight_layout()
plt.show()

Hell's Kitchen makes the most revenue, followed by Astoria and the least performing store being Lower Manhattan. What can be done to imporove their revenue is to run flash sales or limited discounts to encourage foot traffic into the stores.

In [0]:
#Revenue by store (bar chart)
store_rev = df.groupby("store_location")["revenue"].sum().sort_values()
store_rev.plot(kind="barh", figsize=(8,4), title="Total Revenue by Store")
plt.xlabel("Revenue (ZAR)")
plt.tight_layout()
plt.show()

The higest transactions are from 8am to 10am at 17500 transactions, with the peak of day being at 10am as the busiest time of the day.

In [0]:
#Transaction by hour of day
hourly = df.groupby("hour")["transaction_id"].count()
plt.figure(figsize=(12, 5))
plt.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
plt.title('Transactions by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Transactions')
plt.show()

In [0]:
#Create a seaborn correlation heatmap for Unit_price, transaction_qty and revenue
sns.heatmap(df[["unit_price", "transaction_qty", "revenue"]].corr(numeric_only=True, method = "spearman"), annot=True, fmt=".2f", cmap="coolwarm")

In [0]:
#Creating a pivot table
pivot = pd.pivot_table(df,
               index="store_location",
               columns="product_category",
               values="revenue",
               aggfunc="sum")

According to the stacked bar chart below, the store which earns the most is Hell's Kitchen and the product category which drives most revenue is coffee.                               

In [0]:
#Visualize the pivot_table as stacked bar chart
pivot.plot(kind="bar", stacked=True, figsize=(10, 6), edgecolor="white")
plt.title("Transaction Quantity by Store Location and Category (Stacked)")
plt.xlabel("Store Location")
plt.ylabel("Transaction Quantity")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Product Category", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

### Matplotlib
Revenue by product category bar chart with matplotlib library

In [0]:
#Matplotlib by product category
cat_rev = df.groupby("product_category")["revenue"].sum()
cat_rev = cat_rev.sort_values(ascending=False)
cat_rev


In [0]:
#Matplotlib bar graph
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(cat_rev.index, cat_rev.values, color = "#1565c0", edgecolor="white")
ax.tick_params(axis="x", rotation=30)
ax.set_xlabel("Product Category")
ax.set_ylabel("Revenue (ZAR)")
ax.set_title("Revenue by product category")
plt.tight_layout()
plt.show()

### Seaborn
Revenue by product category bar chart with Seaborn library

In [0]:
#Seaborn bar graph
cat_rev = df.groupby("product_category")["revenue"].sum().reset_index()
cat_rev.columns = ["category", "revenue"]
cat_rev = cat_rev.sort_values("revenue",ascending = False)
cat_rev


In [0]:
sns.barplot(data=cat_rev, x="category", y="revenue")
plt.xticks(rotation=30, ha="right")
plt.title("Revenue by product category")
plt.show()

### Plotly
Revenue by product category bar chart with Plotly library

In [0]:
#confirming that plotly is installed
!pip install plotly

In [0]:
import plotly.express as px

In [0]:
cat_rev = df.groupby("product_category")["revenue"].sum().reset_index()
cat_rev.columns = ["category", "revenue"]
cat_rev = cat_rev.sort_values(by='revenue', ascending=False)
cat_rev

In [0]:
fg = px.bar(cat_rev, x="category", y="revenue",
            color="revenue",
            title="Revenue by product category")
fg.show()

### Comparison
- The library that required the most code is Matplotlib.
- The library with the least code required is Plotly.
- The one with the best default output is Plotly library because it is interactive, it shows actual data as you hover the mouse over the graph.
- I would choose plotly over seaborn for presentations of insights to the team, clients or management. 


### Business Insights
- Looking at the revenue by product category graph, it can be seen that the highest product sold is coffee, followed by tea. The least purchased product is packacged chocolate accross all stores.
- Zooming in on the stcaked graph, the bakery sales can be improved by combining coffee and with any bakery product as a combo purchase to increase the sales, and also packaged chocolate could be a good addition to the combo package as a snack.
A combo deal "Hot deal" to incorporate the products described in the previous point to drive sales of the low performing products.
